<a href="https://colab.research.google.com/github/arhadsana/Phenological-wheat-yield-prediction-using-UAV-and-Sentinel-data/blob/main/MLY_Vg01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Colab Notebooks/drone

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#Loading the dataset
df=pd.read_csv('MLYvg.csv',nrows=10000, encoding='ISO-8859-1')

In [ ]:
df.head()

In [ ]:
# Separate features (excluding field columns) and target
X = df.drop(['yield', 'field'], axis=1)  # Replace with actual column names
y = df['yield']  # Replace with actual yield column name

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, LeaveOneOut, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# ------------------------------------------------------------
# 1. Normalize features
# ------------------------------------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
y = np.array(y)

# ------------------------------------------------------------
# 2. hyperparameter grids
# ------------------------------------------------------------

param_grid_rf = {
    'max_depth': [1, 3, 4,],
    'min_samples_leaf': [3, 5]
}

param_grid_ridge = {
    'alpha': [0.001, 0.01, 0.1,]
}

param_grid_lasso = {
    'alpha': [0.001, 0.01, 0.1,]
}

param_grid_svr = {
    'C': [0.1, 1, 5, 10, 20],
            'gamma': ['scale', 'auto', 0.1, 0.01],
            'epsilon': [0.01, 0.1,]
}

param_grid_gbr = {
   'n_estimators': [50, 100, 150],
'learning_rate': [0.01, 0.03, 0.05],
'max_depth': [2, 3, 4, 5],
#'subsample': [0.5, 0.7, 0.9,],
#'max_features': [0.5,0.7, 1.0]
}

# ------------------------------------------------------------
# 3. Initialize models
# ------------------------------------------------------------

rf = RandomForestRegressor(random_state=42)
ridge = Ridge(random_state=42)
lasso = Lasso(random_state=42)
svr = SVR()

# Grid searches
cv = KFold(n_splits=5, shuffle=True, random_state=42)

rf_grid      = GridSearchCV(rf, param_grid_rf,      cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
ridge_grid   = GridSearchCV(ridge, param_grid_ridge,cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
lasso_grid   = GridSearchCV(lasso, param_grid_lasso,cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
svr_grid     = GridSearchCV(svr, param_grid_svr,    cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)

# Fit on the full dataset BEFORE LOOCV
rf_grid.fit(X_scaled, y)
ridge_grid.fit(X_scaled, y)
lasso_grid.fit(X_scaled, y)
svr_grid.fit(X_scaled, y)

rf_best = rf_grid.best_estimator_
ridge_best = ridge_grid.best_estimator_
lasso_best = lasso_grid.best_estimator_
svr_best = svr_grid.best_estimator_

print("\n---------------- Best Hyperparameters ----------------")

print("Random Forest:", rf_grid.best_params_)
print("Ridge:", ridge_grid.best_params_)
print("Lasso:", lasso_grid.best_params_)
print("SVR:", svr_grid.best_params_)
# ------------------------------------------------------------
# 4. LOOCV for generating OOF predictions
# ------------------------------------------------------------
loo = LeaveOneOut()

model_names = ["Random Forest", "Ridge", "Lasso", "SVR", "Ensemble", "GBR Meta-learner"]
all_y_pred = {m: [] for m in model_names}
all_y_true = []
stacked_X_train = []
stacked_y_train = []

for train_index, test_index in loo.split(X_scaled):

    X_train_fold, X_test_fold = X_scaled[train_index], X_scaled[test_index]
    y_train_fold, y_test_fold = y[train_index], y[test_index]

    pred_rf    = rf_best.predict(X_test_fold)[0]
    pred_ridge = ridge_best.predict(X_test_fold)[0]
    pred_lasso = lasso_best.predict(X_test_fold)[0]
    pred_svr   = svr_best.predict(X_test_fold)[0]

    all_y_pred["Random Forest"].append(pred_rf)
    all_y_pred["Ridge"].append(pred_ridge)
    all_y_pred["Lasso"].append(pred_lasso)
    all_y_pred["SVR"].append(pred_svr)

    # First stage ensemble
    ensemble_pred = np.mean([pred_rf, pred_ridge, pred_lasso, pred_svr])
    all_y_pred["Ensemble"].append(ensemble_pred)

    # Save true label
    all_y_true.append(y_test_fold)

    # Meta inputs
    stacked_X_train.append([pred_rf, pred_ridge, pred_lasso, pred_svr])
    stacked_y_train.append(y_test_fold)

stacked_X_train = np.array(stacked_X_train)
stacked_y_train = np.array(stacked_y_train)

# ------------------------------------------------------------
# 5. Train the GBR meta learner
# ------------------------------------------------------------

meta_learner = GradientBoostingRegressor(random_state=42)
gbr_grid = GridSearchCV(meta_learner, param_grid_gbr, cv=cv, scoring='r2', n_jobs=-1)
gbr_grid.fit(stacked_X_train, stacked_y_train)

meta_best = gbr_grid.best_estimator_

print("\nGradient Boosting Meta-learner:", gbr_grid.best_params_)

# Predict meta-learner OOF
meta_oof = meta_best.predict(stacked_X_train)
all_y_pred["GBR Meta-learner"] = meta_oof

# ------------------------------------------------------------
# ------------------------------------------------------------
# 6. Evaluation metrics + Confidence Intervals
# ------------------------------------------------------------

def print_metrics(name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"\n{name}:")
    print(f"  MSE  : {mse}")
    print(f"  RMSE : {rmse}")
    print(f"  MAE  : {mae}")
    print(f"  R²   : {r2}")

# Print metrics for all models
for name in model_names:
    print_metrics(name, all_y_true, all_y_pred[name])


# ------------------------------------------------------------
# 6B. Confidence Intervals using Bootstrap
# ------------------------------------------------------------
from sklearn.utils import resample

def bootstrap_ci(y_true, y_pred, n_bootstrap=2000):
    rmse_list, mae_list, r2_list = [], [], []

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    for _ in range(n_bootstrap):
        idx = resample(range(len(y_true)), replace=True)
        yt = y_true[idx]
        yp = y_pred[idx]

        rmse_list.append(np.sqrt(mean_squared_error(yt, yp)))
        mae_list.append(mean_absolute_error(yt, yp))
        r2_list.append(r2_score(yt, yp))

    return {
        "RMSE_CI": (np.percentile(rmse_list, 2.5), np.percentile(rmse_list, 97.5)),
        "MAE_CI":  (np.percentile(mae_list, 2.5), np.percentile(mae_list, 97.5)),
        "R2_CI":   (np.percentile(r2_list, 2.5), np.percentile(r2_list, 97.5))
    }


# Compute and print CIs for each model
print("\n---------------- Confidence Intervals (95%) ----------------")
for name in model_names:
    ci = bootstrap_ci(all_y_true, all_y_pred[name])
    print(f"\n{name} CIs:")
    print(f"  RMSE 95% CI: {ci['RMSE_CI']}")
    print(f"  MAE  95% CI: {ci['MAE_CI']}")
    print(f"  R²   95% CI: {ci['R2_CI']}")


# ------------------------------------------------------------
# 7. Export predictions for ALL 22 samples
# ------------------------------------------------------------
df_out = pd.DataFrame({
    "Actual": all_y_true,
    "RF": all_y_pred["Random Forest"],
    "Ridge": all_y_pred["Ridge"],
    "Lasso": all_y_pred["Lasso"],
    "SVR": all_y_pred["SVR"],
    "Ensemble": all_y_pred["Ensemble"],
    "Meta_GBR": all_y_pred["GBR Meta-learner"]
})

df_out.to_csv("LOOCV_predictions_Vg.csv", index=False)
print("\nExported → LOOCV_predictions_Vg.csv")


In [ ]:
import matplotlib.pyplot as plt
model_list = ["Random Forest","Ridge","Lasso","SVR","Ensemble","GBR Meta-learner"]

rmse_means = []
rmse_err_lower = []
rmse_err_upper = []

for name in model_list:
    ci = bootstrap_ci(all_y_true, all_y_pred[name])
    rmse_low, rmse_high = ci["RMSE_CI"]

    rmse_mean = np.sqrt(mean_squared_error(all_y_true, all_y_pred[name]))
    rmse_means.append(rmse_mean)
    rmse_err_lower.append(rmse_mean - rmse_low)
    rmse_err_upper.append(rmse_high - rmse_mean)

plt.figure(figsize=(10,6))
plt.bar(model_list, rmse_means, yerr=[rmse_err_lower, rmse_err_upper],
        capsize=5, color="pink")

plt.ylabel("RMSE")
plt.title("Model Performance with 95% Confidence Intervals")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("rmse_CI_barplot.png", dpi=300)
plt.show()


In [ ]:
import shap

# ------------------------------------------------------------
# 8. SHAP for original indices on the stacked meta-learner
# ------------------------------------------------------------

# If X is a DataFrame, keep feature names
try:
    feature_names = list(X.columns)
except AttributeError:
    feature_names = [f"X{i}" for i in range(X_scaled.shape[1])]

# Define a prediction function for the *entire* stacked model
# It takes SCALED features as input and outputs meta-learner predictions
def stacked_model_predict(X_scaled_input):
    # base model predictions
    rf_p    = rf_best.predict(X_scaled_input)
    ridge_p = ridge_best.predict(X_scaled_input)
    lasso_p = lasso_best.predict(X_scaled_input)
    svr_p   = svr_best.predict(X_scaled_input)

    # stack them as meta-features
    stacked = np.column_stack([rf_p, ridge_p, lasso_p, svr_p])

    # meta-learner output
    return meta_best.predict(stacked)

# We use the full (scaled) dataset as background & evaluation points
background = X_scaled   # shape (22, n_features)

# KernelExplainer works for any black-box regression model
explainer_indices = shap.KernelExplainer(stacked_model_predict, background)

# SHAP values for all samples and all original indices
shap_values_indices = explainer_indices.shap_values(X_scaled)

# Convert to DataFrame for export
shap_indices_df = pd.DataFrame(shap_values_indices, columns=feature_names)
shap_indices_df["Actual_yield"] = y
shap_indices_df.to_csv("SHAP_indices_meta_GBR_Rp.csv", index=False)
print("Exported → SHAP_indices_meta_GBR_Rp.csv")

# Summary plots (for figures)
shap.summary_plot(shap_values_indices, X_scaled, feature_names=feature_names)
shap.summary_plot(shap_values_indices, X_scaled, feature_names=feature_names, plot_type="bar")
